In [ ]:
# Packages pypowsybl
import pypowsybl as pp
import pypowsybl.network as pn
import pypowsybl.loadflow as lf
# Detail logging packages (knitro)
import logging

import pypowsybl.loadflow as lf


# vizualization packages 
from slack_viz_utils import nad_explorer_with_slack, calculate_dc_losses
import pandas as pd


KNITRO SOLVER, with Pypowsybl

Input parameter is a network that does not converges with Newton-Raphson Method 

In [ ]:
# Load the non convergent network 
Path_to_network = r"Outputs\ieee14-voltage-perturbation.xiidm"
network_KN = pn.load(Path_to_network)
network_NR = pn.load(Path_to_network)
network_DC = pn.load(Path_to_network)

In [ ]:
# Newton Raphson solver diverges 
p_NR = lf.Parameters(provider_parameters={'acsolverType': 'NEWTON_RAPHSON'})
lf.run_ac(network_NR, p_NR )

In [ ]:
# Run a DC LF on the perturbated network to approximate losses, and for a weight for the objective function 
lf.run_dc(network_DC)
voltage_level_df = network_DC.get_voltage_levels()
losses = calculate_dc_losses(network_DC, voltage_level_df)

print(f"DC LOSSES Approximation : {losses:.2f} MW")

In [ ]:
# Parameters to choose the different level of logging details : DEBUG, INFO 
logging.basicConfig()
logging.getLogger('powsybl').setLevel(logging.DEBUG)

In [ ]:
Path_to_CSV = r'TEST1_Result\ieee14-perturbation'
p_KN = lf.Parameters( distributed_slack=False, use_reactive_limits=False, provider_parameters={
  'acSolverType': 'KNITRO', 'solverType':'RELAXED', 'losses': str(losses), 
  'maxKnitroIterations': '200', 'gradientComputationMode': '1', 'threadNumber':'1', 
  'gradientUserRoutine': '2', 'hessianComputationMode': '6', 'minRealisticVoltage': '0.5', 
  'maxRealisticVoltage': '1.5', 'slackThreshold':'0.000001', 
  'relativeFeasibilityStoppingCriteria': '0.000001', 'absoluteFeasibilityStoppingCriteria':'0.001',
  'relativeOptimalityStoppingCriteria': '0.000001', 'absoluteOptimalityStoppingCriteria': ' 0.001', 'optimalityStoppingCriteria':'0.0000001', 
  'alwaysUpdateNetwork': 'false','exportSolution': Path_to_CSV })
lf.run_ac(network_KN, p_KN)

Load the slack's informations from the export solution 

In [ ]:
data = pd.read_csv(Path_to_CSV + ".csv", sep=";")
data

In [ ]:
from pypowsybl.network import NadParameters
# use the nad_explorer_with_slack function, a personalized function to vizualized and interpret the solver's result
explorer = nad_explorer_with_slack(network_KN, slack_info=data, outerloop=0)
explorer